Name: Fahmi Sajid Ahmed

Student Number: 300250180

In [7]:
# Import Libraries + required assets

import pandas as pd;
import numpy as np;
from sklearn.preprocessing import StandardScaler;
from sklearn.impute import KNNImputer;

url = "https://raw.githubusercontent.com/Fahmi-IT/CSI4142_A2/refs/heads/a2_part2/global_tech_fundamentals_2026.csv";
df = pd.read_csv(url);

numeric_cols = [
    "Market_Cap_B", "Employees", "Total_Revenue_B", "Research_And_Development_B",
    "Net_Income_B", "Total_Debt_B", "Free_Cash_Flow_B", "R&D_to_Revenue_Pct",
    "Gross_Margin_Pct", "Operating_Margin_Pct", "Profit_Margin_Pct",
    "Trailing_PE", "Forward_PE", "Price_to_Book", "Beta_Volatility",
    "Institutional_Ownership_Pct", "Short_Float_Pct", "Dividend_Yield_Pct",
    "52_Week_Change_Pct"
];

Dataset URL: https://www.kaggle.com/datasets/kanchana1990/global-tech-giants-fundamentals-and-valuations-2026

This is CSI4142 A2. The purpose of this portion of the assignment is to explore various imputation methods.

The dataset used describes a deep cross-sectional snapshot of the core business mechanics of the top 100+ tech, semiconductor, and software companies in the world. The author, Kanchana Gajamuthu, engineered the dataset specifically for Exploratory Data Analysis so that data scientists and AI engineers alike could use it (not for a practical purpose).

The attributes are as follows:
- **Ticker**: The stock ticker symbol used to identify the company on public exchanges.
- **Company_Name**: The full legal name of the company.
- **Industry**: The primary industry sector in which the company operates.
- **Market_Cap_B**: The company’s total market capitalization in billions of dollars.
- **Employees**: The total number of employees working for the company.
- **Total_Revenue_B**: The company’s total annual revenue in billions of dollars.
- **Research_And_Development_B**: The annual amount spent on research and development in billions of dollars.
- **Net_Income_B**: The company’s net profit after expenses in billions of dollars.
- **Total_Debt_B**: The total outstanding debt held by the company in billions of dollars.
- **Free_Cash_Flow_B**: The company’s free cash flow in billions of dollars.
- **R&D_to_Revenue_Pct**: The percentage of revenue allocated to research and development.
- **Gross_Margin_Pct**: The percentage of revenue remaining after cost of goods sold.
- **Operating_Margin_Pct**: The percentage of revenue remaining after operating expenses.
- **Profit_Margin_Pct**: The percentage of revenue that translates into net profit.
- **Trailing_PE**: The price-to-earnings ratio based on past (historical) earnings.
- **Forward_PE**: The price-to-earnings ratio based on projected future earnings.
- **Price_to_Book**: The ratio of the company’s market value to its book value.
- **Beta_Volatility**: A measure of the stock’s volatility relative to the overall market.
- **Institutional_Ownership_Pct**: The percentage of shares owned by institutional investors.
- **Short_Float_Pct**: The percentage of shares available for trading that are currently sold short.
- **Dividend_Yield_Pct**: The annual dividend yield expressed as a percentage of the stock price.
- **52_Week_Change_Pct**: The percentage change in the stock price over the past 52 weeks.

1.1

Type of Imputation: Default Value Imputation (Median)

Which attribute is being impacted: Free_Cash_Flow_B

What is Default Value Imputation (Median)?

Default Value Imputation (Median) is an imputation method that involves replacing missing values with the median value of that attribute in the dataset.

1.2

The column Free_Cash_Flow_B was chosen to undergo MCAR removal because it is a continuous financial metric that tends to be right-skewed due to the tech giant outliers (Apple, Microsoft, etc.), which makes the median a better default than, say, the mean. A predetermined number of values were removed at random using the np.random.choice() function and replacing the selected indices with NaN.

In [8]:
# 1.3
TARGET = "Free_Cash_Flow_B";

MISSING_RATE = 0.15;  # 15% of values will be removed (MCAR)

# Remove values according to MCAR method (replace with NaN)
df_mcar = df.copy();
missing_indices = np.random.choice(df_mcar.index, size=int(len(df_mcar) * MISSING_RATE), replace=False);
df_mcar.loc[missing_indices, TARGET] = np.nan;

# Default Value Imputation (Median)
median_value = df_mcar[TARGET].median();
print(f"Computed median  : {median_value:.4f}");
df_imputed = df_mcar.copy();
df_imputed[TARGET] = df_imputed[TARGET].fillna(median_value);

# Summary of Changes
print(f"\nStat Summary");
summary = pd.DataFrame({
    "Original"  : df[TARGET].describe(),
    "After MCAR": df_mcar[TARGET].describe(),
    "Imputed"   : df_imputed[TARGET].describe(),
});
print(summary.to_string());

# Save MCAR removed + median imputed versions to file
df_mcar.to_csv("tech_mcar.csv", index=False);
df_imputed.to_csv("tech_median_imputed.csv", index=False);
print("\nFiles saved: tech_mcar.csv, tech_median_imputed.csv");

Randomly selected attribute : Free_Cash_Flow_B
Computed median  : 1.3450

Stat Summary
         Original  After MCAR     Imputed
count  103.000000   88.000000  103.000000
mean    20.528835   22.364091   19.303058
std    100.330298  108.083509  100.097983
min     -4.950000   -4.950000   -4.950000
25%      0.590000    0.622500    0.730000
50%      1.360000    1.345000    1.345000
75%      6.410000    6.245000    4.590000
max    992.380000  992.380000  992.380000

Files saved: tech_mcar.csv, tech_median_imputed.csv


1.4
Evaluation Here

2.1

Type of Imputation: Correlation Imputation

Which attribute is being impacted: fulfillment_time_min

Correlation Imputation is an imputation method that estimates missing values via a correlation matrix with another numerical feature (in this case, Total_Revenue_B) (larger revenue = more employees).

2.2

MAR removal was used on the fulfillment_time_min attribute as hwardware, consumer electronics, and internet retail companies are given much higher rates of missing values in comparison to software companies, which reflects a realistic MAR scenario. This was done by assigning each industry a specific probability of having its Employees value removed (ranging from 80% for Computer Hardware to 5% for Software - Application) and then randomly selecting a subset of those rows proportional to the assigned probability set to NaN.

In [9]:
# 2.3
TARGET = "fulfillment_time_min";

# MAR Removal
industry_missing_prob = {
    "Computer Hardware"                     : 0.80,
    "Consumer Electronics"                  : 0.80,
    "Internet Retail"                       : 0.60,
    "Semiconductor Equipment & Materials"   : 0.50,
    "Electronic Components"                 : 0.50,
    "Semiconductors"                        : 0.30,
    "Information Technology Services"       : 0.20,
    "Software - Application"                : 0.05,
    "Software - Infrastructure"             : 0.05,
    "Internet Content & Information"        : 0.05,
    "Credit Services"                       : 0.05,
    "Entertainment"                         : 0.05,
    "Communication Equipment"               : 0.05,
    "Travel Services"                       : 0.05,
    "Electronic Gaming & Multimedia"        : 0.05,
    "Specialty Business Services"           : 0.05,
    "Leisure"                               : 0.05,
};

df_mar = df.copy();
missing_mask = pd.Series(False, index=df_mar.index);

for industry, prob in industry_missing_prob.items():
    idx = df_mar[df_mar["Industry"] == industry].index;
    if len(idx) == 0:
        continue;
    n_remove = max(1, int(len(idx) * prob)) if prob >= (1 / len(idx)) else 0;
    chosen = np.random.choice(idx, size=min(n_remove, len(idx)), replace=False);
    missing_mask.loc[chosen] = True;

df_mar.loc[missing_mask, TARGET] = np.nan;

# Verify MAR Property
print(f"\nMissing rate by Industry:");
mar_check = (
    df_mar.groupby("Industry")[TARGET]
    .apply(lambda x: f"{x.isna().mean() * 100:.0f}%  ({x.isna().sum()}/{len(x)})")
    .sort_values(ascending=False)
);
print(mar_check.to_string());

PREDICTOR = "Total_Revenue_B";
complete_mask = df_mar[TARGET].notna();
df_complete   = df_mar[complete_mask];
df_missing    = df_mar[~complete_mask];
X_train = df_complete[[PREDICTOR]].values;
y_train = df_complete[TARGET].values;
model = LinearRegression();
model.fit(X_train, y_train);

X_missing   = df_missing[[PREDICTOR]].values;
predictions = np.maximum(model.predict(X_missing), 0);  # employees can't be negative
df_imputed = df_mar.copy();
df_imputed.loc[~complete_mask, TARGET] = predictions.round().astype(int);

print(f"\nStats Summary");
summary = pd.DataFrame({
    "Original" : df[TARGET].describe(),
    "After MAR": df_mar[TARGET].describe(),
    "Imputed"  : df_imputed[TARGET].describe(),
});
print(summary.to_string());

original_at_missing = df.loc[~complete_mask, TARGET];
imputed_at_missing  = df_imputed.loc[~complete_mask, TARGET];

mae  = np.mean(np.abs(original_at_missing - imputed_at_missing));
rmse = np.sqrt(np.mean((original_at_missing - imputed_at_missing) ** 2));
print(f"\nImputation Quality (vs ground truth)");
print(f"MAE  : {mae:,.0f} employees");
print(f"RMSE : {rmse:,.0f} employees");

# Save MAR removed + correlation imputed versions to file
df_mar.to_csv("tech_mar.csv", index=False);
df_imputed.to_csv("tech_correlation_imputed.csv", index=False);
print("\nFiles saved: tech_mar.csv, tech_correlation_imputed.csv");

3.1

Type of Imputation: Similarity-based Imputation

Which attribute is being impacted: Dividend_Yield_Pct

Similaity-based imputation basically uses the similarity between records in the dataset to estimate missing ones. A common application of this is kNN.

3.2

MNAR removal was applied to the Dividend_Yield_Pct attribure because the missingness can be tied directly to the value itself in 3 tiers: zero-dividend companies, low-yield companies, and high-yield companies. This helps represent a realistic MNAR scenario (those with nothing good to show won't show it). This was done by essentially generating a random continuous value between 0 and 1. If the generated value fell below the threshold according to the row's tier classification, then the attribute value for that row would be set to NaN.

In [ ]:
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

used_previously = ["Free_Cash_Flow_B", "Employees"]
eligible = [c for c in numeric_cols if c not in used_previously]
TARGET = np.random.choice(eligible)
print(f"Randomly selected attribute : {TARGET}")

df_mnar = df.copy()
missing_mask = pd.Series(False, index=df_mnar.index)
for idx, row in df_mnar.iterrows():
    val = row[TARGET]
    if val == 0:
        prob = 0.85
    elif val <= 50:
        prob = 0.40
    else:
        prob = 0.05
    if np.random.rand() < prob:
        missing_mask.loc[idx] = True
df_mnar.loc[missing_mask, TARGET] = np.nan

print(f"\nMissing rate by value tier (ground truth):")
bins   = [-1, 0, 50, float("inf")]
labels = ["Zero (0)", "Low (1–50)", "High (>50)"]
tiers  = pd.cut(df[TARGET], bins=bins, labels=labels)
mnar_check = pd.DataFrame({"tier": tiers, "is_missing": missing_mask})
print(mnar_check.groupby("tier", observed=True)["is_missing"]
      .apply(lambda x: f"{x.mean()*100:.0f}%  ({x.sum()}/{len(x)})")
      .to_string())

KNN_FEATURES = [
    "Gross_Margin_Pct",
    "Forward_PE",
    "Beta_Volatility",
    "Operating_Margin_Pct",
    TARGET
]
K = 5  # number of neighbours
scaler      = StandardScaler()
knn_data    = df_mnar[KNN_FEATURES].copy()
feature_cols = KNN_FEATURES[:-1]

knn_data[feature_cols] = scaler.fit_transform(knn_data[feature_cols])

# Run KNN imputer
imputer    = KNNImputer(n_neighbors=K, weights="distance")
imputed_arr = imputer.fit_transform(knn_data)

# Rebuild dataframe and reverse standardisation on features (TARGET was never scaled)
imputed_knn = pd.DataFrame(imputed_arr, columns=KNN_FEATURES, index=df_mnar.index)

df_imputed = df_mnar.copy()
df_imputed[TARGET] = imputed_knn[TARGET]

print(f"\nStats Summary")
summary = pd.DataFrame({
    "Original"  : df[TARGET].describe(),
    "After MNAR": df_mnar[TARGET].describe(),
    "Imputed"   : df_imputed[TARGET].describe(),
})
print(summary.to_string())

original_at_missing = df.loc[missing_mask, TARGET]
imputed_at_missing  = df_imputed.loc[missing_mask, TARGET]

mae  = np.mean(np.abs(original_at_missing - imputed_at_missing))
rmse = np.sqrt(np.mean((original_at_missing - imputed_at_missing) ** 2))
print(f"\n── Imputation Quality (vs ground truth) ──────────────────────────────────")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

# Save MNAR removed + similarity imputed versions to file
df_mnar.to_csv("tech_mnar.csv", index=False)
df_imputed.to_csv("tech_knn_imputed.csv", index=False)
print("\nFiles saved: tech_mnar.csv, tech_knn_imputed.csv")

References

Scikit-learn KNN Imputer Documentation
https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html